In [1]:
MAX_COORD = 5.12
MIN_COORD = -5.12

MIN_COORD_S = -65.536
MAX_COORD_S = 65.536

sigma = (MAX_COORD - MIN_COORD) / 10
sigma_s = (MAX_COORD_S - MIN_COORD_S) / 10
import random

In [2]:

def complete_average_crossover(parent1, parent2) :
    """This crossover is done by taking the mean of the 2 parents and creating just one child

    Args:
        parent1,2 (list[float]):  the parents chosen for the crossover

    Returns:
        child (list[float]) : the result of the crossover
    """
    return [(parent1[i] + parent2[i]) / 2 for i in range(len(parent1))]

In [3]:
def arithmetic_crossover(parent1, parent2):
    """This crossover is done by simulation taking percentages out of the parents for each children

    Args:
        parent1,2 (list[float]): the parentss chosen for the crossover 
        crossover_rate (float): rate of each gene beeing mutated

    Returns:
        (child1, child2) (tuple(list[float], list[float])) : the children resulted from the crossover
    """
    crossover_rate = 0.8
    
    if random.random() > crossover_rate:
        return list(parent1), list(parent2)
    
    alpha = random.random()
    child1 = [alpha * parent1[i] + (1 - alpha) * parent2[i] for i in range(len(parent1))]
    child2 = [(1 - alpha) * parent1[i] + alpha * parent2[i] for i in range(len(parent1))]
    return child1, child2

In [4]:
def uniform_mutation(individual, lower, upper) :
    """Mutation done by changing for each gene with a mutation_rate chance the value of the gene with one 
    randomly chosen from the uniform distribution from the MIN_COORD and MAX_COORD

    Args:
        individual (list[float]): the solution to be changed
        mutatution_rate (float): rate of mutation happening for each gene

    Returns:
        individual (list[float]) : mutated solution
    """
    mutatution_rate = 0.01
    for i in range(len(individual)):
        if random.random() < mutatution_rate:
            individual[i] = random.uniform(lower, upper)
    return individual

In [5]:
import random

def gaussian_mutation(individual, sigma, lower, upper):
    """
    Normally distributed mutation for real-valued representations.
    Each gene is perturbed by adding a sample from N(0, sigma).
    
    Args:
        individual: list of real-valued genes
        sigma: standard deviation of the Gaussian — controls step size
        lower: lower bound of search space
        upper: upper bound of search space
    
    Returns:
        mutated individual
    """
    mutated = individual.copy()
    for i in range(len(mutated)):
        mutated[i] += random.gauss(0, sigma)
        mutated[i] = max(lower, min(upper, mutated[i]))
    return mutated

In [6]:
def sphere_fitness_function(individual) :
    """Fitness function for the sphere problem

    Args:
        individual (list[float]): solution

    Returns:
        (float): fitness of solution
    """
    return sum(x**2 for x in individual)

In [7]:
def schwefel_fitness_function(individual : list[float]) -> float :
    """Fitness function for the schwefel problem

    Args:
        individual (list[float]): solution

    Returns:
        sum (float): fitness of solution
    """
    sum = 0
    
    for i in range(len(individual)) :
        for j in range(i+1) :
            sum+=individual[j]**2
            
    return sum    

In [8]:
def tournament_selection(population, tournament_size, dataset) :
    """Strategy of time tournament for choosing parents to reproduce

    Args:
        population (list[list[float]]): population of current solutions
        tournament_size (_type_): nr of solutions chosen to enter tournament
        dataset (_type_): type of fitness function to use

    Returns:
        best_parent (list[float]) : solution that wins the tournament
    """
    if dataset == 'sphere' :
        return min(random.sample(population, tournament_size), key=lambda individual : sphere_fitness_function(individual))
    
    if dataset == 'schwefel' :
        return min(random.sample(population, tournament_size), key=lambda individual : schwefel_fitness_function(individual))
        

In [ ]:
def roulette_wheel_selection(population, dataset):
    """
    Proportional selection (roulette wheel) as defined in slides.
    Probability of selecting individual i = f(xi) / F
    where F = sum of all fitnesses.
    
    Since we minimize, fitness is inverted so better solutions
    (lower values) get higher selection probability.
    
    Args:
        population: list of (individual) 
    
    Returns:
        selected individual
    """
    if dataset == 'sphere' :
        pop_with_fitness = [(individual, sphere_fitness_function(individual)) for individual in population]
    if dataset == 'schwefel' :
        pop_with_fitness = [(individual, schwefel_fitness_function(individual)) for individual in population]
        
    max_fit = max(f for _, f in pop_with_fitness)
    inverted = [(ind, max_fit - f + 1e-10) for ind, f in pop_with_fitness]
    
    F = sum(f for _, f in inverted)
    
    cumulative = []
    running = 0
    for _, f in inverted:
        running += f / F
        cumulative.append(running)
    
    g = random.random()
    
    for i, q in enumerate(cumulative):
        if g <= q:
            return inverted[i][0]
    
    return inverted[-1][0] 

In [10]:
import random


def ea_for_sphere_function(iterations, population_size, 
                        crossover_type, mutation_type, 
                        selection_type, dimension = 3) :
    
    population = [[random.uniform(MIN_COORD, MAX_COORD) for _ in range(dimension)] for _ in range(population_size)]
    
    
    for _ in range(iterations) :
        
        new_population = []
        sorted_population = sorted(population, key=lambda individual : sphere_fitness_function(individual))
        new_population += sorted_population[:10]
        
        while len(new_population) < population_size :
            
            if selection_type == 'tournament' :
                parent1 = tournament_selection(population, 3, dataset='sphere')
                parent2 = tournament_selection(population, 3, dataset = 'sphere')
            
            if selection_type == 'roullete' :
                parent1 = roulette_wheel_selection(population, dataset = 'sphere')
                parent2 = roulette_wheel_selection(population, dataset ='sphere')
            
            children = []
            
            if crossover_type == 'complete_average' :
                child = complete_average_crossover(parent1, parent2)
                children.append(child)
            
            if crossover_type == 'arithmetic' :
                child1, child2 = arithmetic_crossover(parent1, parent2)
                children.append(child1)
                children.append(child2)
            
            
            for childk in children :
                if len(new_population) < population_size :
                    if mutation_type == 'uniform' :
                        new_population.append(uniform_mutation(childk, MIN_COORD, MAX_COORD))
                    if mutation_type == 'gaussian' :
                        new_population.append(gaussian_mutation(childk, sigma, MIN_COORD, MAX_COORD))
        
        population = new_population
            
            
    return min(population, key=lambda individual : sphere_fitness_function(individual))

In [11]:
import random


def ea_for_schwefel_function(iterations, population_size, 
                        crossover_type, mutation_type, 
                        selection_type, dimension = 3) :
    
    population = [[random.uniform(MIN_COORD_S, MAX_COORD_S) for _ in range(dimension)] for _ in range(population_size)]
    
    
    for _ in range(iterations) :
        
        new_population = []
        sorted_population = sorted(population, key=lambda individual : schwefel_fitness_function(individual))
        new_population += sorted_population[:10]
        
        while len(new_population) < population_size :
            
            if selection_type == 'tournament' :
                parent1 = tournament_selection(population, 3, dataset = 'schwefel')
                parent2 = tournament_selection(population, 3, dataset = 'schwefel')
            
            if selection_type == 'roullete' :
                parent1 = roulette_wheel_selection(population, dataset='schwefel')
                parent2 = roulette_wheel_selection(population, dataset='schwefel')
            
            children = []
            
            if crossover_type == 'complete_average' :
                child = complete_average_crossover(parent1, parent2)
                children.append(child)
            
            if crossover_type == 'arithmetic' :
                child1, child2 = arithmetic_crossover(parent1, parent2)
                children.append(child1)
                children.append(child2)
            
            
            for childk in children :
                if len(new_population) < population_size :
                    if mutation_type == 'uniform' :
                        new_population.append(uniform_mutation(childk, MIN_COORD_S, MAX_COORD_S))
                    if mutation_type == 'gaussian' :
                        new_population.append(gaussian_mutation(childk, sigma_s, MIN_COORD_S, MAX_COORD_S))
        
        population = new_population
            
            
    return min(population, key=lambda individual : schwefel_fitness_function(individual))

In [12]:
def generate_report_sphere(output: str,
                    iterations_values, population_values,
                    crossover_type, mutation_type, selection_type):
    """The function that generates the reports.
    """

    with open(output, 'a') as f:
        f.write(f"\nReport for sphere\n")

    for iteration in iterations_values :
        for population in population_values :
            for crossover in crossover_type :
                for mutation in mutation_type :
                    for selection in selection_type :
                        result = ea_for_sphere_function(iteration, population,
                                                        crossover, mutation, selection, 3)
    
                        with open(output, 'a') as f:
                            f.write(f"iterations={iteration}, population={population}, crossover={crossover}, mutation={mutation}, selection={selection} => result={result}\n")

In [13]:
def generate_report_schwefel(output: str,
                    iterations_values, population_values,
                    crossover_type, mutation_type, selection_type):
    """The function that generates the reports.
    """

    with open(output, 'a') as f:
        f.write(f"\nReport for schwefel\n")

    for iteration in iterations_values :
        for population in population_values :
            for crossover in crossover_type :
                for mutation in mutation_type :
                    for selection in selection_type :
                        result = ea_for_schwefel_function(iteration, population,
                                                        crossover, mutation, selection, 3)
    
                        with open(output, 'a') as f:
                            f.write(f"iterations={iteration}, population={population}, crossover={crossover}, mutation={mutation}, selection={selection} => result={result}\n")

In [14]:
generate_report_sphere(
    output='report_sphere.txt',
    iterations_values=[100, 500, 1000],
    population_values=[50, 100],
    crossover_type=['complete_average', 'arithmetic'],
    mutation_type=['uniform', 'gaussian'],
    selection_type=['tournament', 'roullete']
)

In [15]:
generate_report_schwefel(
    output='report_schwefel.txt',
    iterations_values=[100, 500, 1000],
    population_values=[50, 100],
    crossover_type=['complete_average', 'arithmetic'],
    mutation_type=['uniform', 'gaussian'],
    selection_type=['tournament', 'roullete']
)